# Test notebook for preprocessing.py

Goal: sanity-check the preprocessing module before using it in the actual training notebook.
We check:
1. The module imports correctly
2. Class names look right and images/labels are actually matched up correctly
3. Split sizes are stratified as expected
4. Shapes and value ranges after normalization
5. Augmentation is visibly doing something on the training set

## 0. Setup: make src/ importable

Adjust the relative path below depending on where this notebook lives
relative to `src/gluten_guard/`. If this notebook is in `notebooks/`, and
`preprocessing.py` lives in `src/gluten_guard/`, the path below should work.

In [ ]:
import sys
from pathlib import Path

# Add src/ to the Python path so we can import gluten_guard as a package
SRC_PATH = Path("../src").resolve()
if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

print("Added to path:", SRC_PATH)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Adjust the import below to match your actual package name
# (e.g. gluten_guard.preprocessing, if preprocessing.py sits inside that package)
from gluten_guard.preprocessing import load_datasets, build_pipeline, IMG_SIZE, BATCH_SIZE

DATA_DIR = "../data/raw/food-101-5-classes"  # adjust if your path differs

## 1. Load the datasets and inspect class names

In [ ]:
train_ds, val_ds, test_ds, class_names = load_datasets(DATA_DIR)

print(f"Number of classes: {len(class_names)}")
print("Class names:", class_names)

## 2. Check split sizes

Roughly 70% train / 15% val / 15% test, in terms of number of BATCHES here
(each dataset is already batched).

In [ ]:
def count_images(ds):
    # Sum up batch sizes across the whole dataset to get the total image count
    total = 0
    for images, _ in ds:
        total += images.shape[0]
    return total

n_train = count_images(train_ds)
n_val = count_images(val_ds)
n_test = count_images(test_ds)
n_total = n_train + n_val + n_test

print(f"Train: {n_train} ({n_train / n_total:.1%})")
print(f"Val:   {n_val} ({n_val / n_total:.1%})")
print(f"Test:  {n_test} ({n_test / n_total:.1%})")

## 3. Visual check: do images actually match their labels?

This is the important sanity check for your question about how images get
connected to their class. If preprocessing.py is working correctly, the
title above each image (taken from the label) should visibly match what's
actually in the picture (e.g. a picture of baklava should be titled
"baklava", not "pizza").

In [ ]:
# Take one batch from the RAW train_ds (before normalization/augmentation)
# so the images displayed still have their original [0, 255] pixel values.
images, labels = next(iter(train_ds))

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, image, label in zip(axes.flat, images, labels):
    ax.imshow(image.numpy().astype("uint8"))
    ax.set_title(class_names[label.numpy()])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Check shapes and value ranges after the full pipeline

In [ ]:
train_ds_processed, val_ds_processed, test_ds_processed = build_pipeline(train_ds, val_ds, test_ds)

images, labels = next(iter(train_ds_processed))

print("Image batch shape:", images.shape)  # expect (BATCH_SIZE, 224, 224, 3)
print("Expected image size:", IMG_SIZE)
print("Expected batch size:", BATCH_SIZE)
print("Pixel value range after normalization:", float(images.numpy().min()), "-", float(images.numpy().max()))
print("Label batch shape:", labels.shape)
print("Sample labels:", labels.numpy()[:8])

## 5. Visual check: is augmentation actually doing something?

Run the same image through the augmentation pipeline multiple times --
you should see visible differences each time (flip, rotation, zoom, contrast).

In [ ]:
from gluten_guard.preprocessing import data_augmentation

# Grab a single raw (unnormalized) image to augment repeatedly
raw_images, raw_labels = next(iter(train_ds))
single_image = raw_images[0:1]  # keep batch dimension
single_label = class_names[raw_labels[0].numpy()]

fig, axes = plt.subplots(1, 5, figsize=(16, 4))
axes[0].imshow(single_image[0].numpy().astype("uint8"))
axes[0].set_title(f"original ({single_label})")
axes[0].axis("off")

for i in range(1, 5):
    augmented = data_augmentation(single_image, training=True)
    axes[i].imshow(augmented[0].numpy().astype("uint8"))
    axes[i].set_title(f"augmented #{i}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

## Summary

If all cells above ran without errors and:
- class names look correct,
- split percentages are roughly 70/15/15,
- images visually match their printed labels,
- pixel values after normalization are in [0, 1],
- augmented images visibly differ from the original,

...then `preprocessing.py` is working as intended and you can safely import
it into your actual training notebook.